# 🛡️ Safeguarding Companion
## RAG-Based Policy Question-Answering System

**Pipeline Overview:**
- ✅ Step 0: Install dependencies (pinned transformers==4.40.0)
- ✅ Step 1: Download PDFs from GitHub & chunk documents
- ✅ Step 2: Generate embeddings
- ✅ Step 3: Hybrid retrieval
- ✅ Step 4: Generate simplified answers (flan-t5-large)
- ✅ Step 5: Gradio UI with voice input & text-to-speech output

---
## 📦 Step 0: Install Dependencies

> ⚠️ **Important:** We pin `transformers==4.40.0` because newer versions removed `text2text-generation` support for flan-t5.
> After this cell finishes, go to **Runtime → Restart Runtime**, then run all cells again.

In [18]:
# First, ensure all conflicting major libraries are uninstalled to start fresh.
# This helps resolve complex dependency issues where pip might not downgrade.
!pip uninstall -y transformers sentence-transformers huggingface-hub PyPDF2 scikit-learn nltk gradio torch requests gTTS pytesseract pdf2image

# Install core dependencies with specific versions to resolve conflicts.
# Pinning huggingface_hub to 0.34.0 (compatible with most recent libs like gradio/tokenizers).
# Pinning sentence-transformers to 2.7.0 (compatible with transformers 4.40.0 and modern huggingface_hub).
!pip install -q "huggingface_hub==0.34.0"
!pip install -q "transformers==4.40.0" # Critical for flan-t5 text2text-generation
!pip install -q "sentence-transformers==2.7.0" # Compatible with transformers 4.40.0 and huggingface_hub >=0.14.0
!pip install -q "requests==2.32.4" # Required by google-colab
!pip install -q "torch" # Install torch
!pip install -q "scikit-learn" # Install scikit-learn

# Install other Python dependencies.
!pip install -q PyPDF2 nltk gradio gTTS pytesseract pdf2image

# Install system-level dependencies for PDF processing (poppler-utils) and OCR (tesseract-ocr).
!apt-get update -qq
!apt-get install -y poppler-utils tesseract-ocr

print("")
print("✅ All packages installed.")
print("⚠️  NOW: Runtime → Restart Runtime, then run all cells again.")

Found existing installation: transformers 4.40.0
Uninstalling transformers-4.40.0:
  Successfully uninstalled transformers-4.40.0
Found existing installation: sentence-transformers 2.2.0
Uninstalling sentence-transformers-2.2.0:
  Successfully uninstalled sentence-transformers-2.2.0
Found existing installation: huggingface_hub 0.36.2
Uninstalling huggingface_hub-0.36.2:
  Successfully uninstalled huggingface_hub-0.36.2
Found existing installation: PyPDF2 3.0.1
Uninstalling PyPDF2-3.0.1:
  Successfully uninstalled PyPDF2-3.0.1
Found existing installation: scikit-learn 1.8.0
Uninstalling scikit-learn-1.8.0:
  Successfully uninstalled scikit-learn-1.8.0
Found existing installation: nltk 3.9.4
Uninstalling nltk-3.9.4:
  Successfully uninstalled nltk-3.9.4
Found existing installation: gradio 6.14.0
Uninstalling gradio-6.14.0:
  Successfully uninstalled gradio-6.14.0
Found existing installation: torch 2.10.0
Uninstalling torch-2.10.0:
  Successfully uninstalled torch-2.10.0
Found existing in

---
## 📄 Step 1: Download PDFs from GitHub & Process Documents

In [ ]:
import os
import re
import requests
import PyPDF2
import numpy as np
import pandas as pd
import nltk
from pdf2image import convert_from_path
import pytesseract

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize


# ── GITHUB PDF LINKS ─────────────────────────────────────────────────────────────
# ✅ Updated to user's GitHub repository
GITHUB_PDF_URLS = [
    "https://raw.githubusercontent.com/madrinejean123/MACHINE-LEARNING--CHATBOT-GROUP-O/madrine/data/Makerere-Safeguarding-Policy.pdf",
    "https://raw.githubusercontent.com/madrinejean123/MACHINE-LEARNING--CHATBOT-GROUP-O/madrine/data/Policy-and-Regulations-Against-Sexual-Harassment-2018.pdf",
    "https://raw.githubusercontent.com/madrinejean123/MACHINE-LEARNING--CHATBOT-GROUP-O/madrine/data/Makerere-Policy-on-Persons-Living-With-Disabilities.pdf",
    "https://raw.githubusercontent.com/madrinejean123/MACHINE-LEARNING--CHATBOT-GROUP-O/madrine/data/FINAL-REVISED-NATIONAL-POLICY-ON-PWDs-2023.pdf",
    "https://raw.githubusercontent.com/madrinejean123/MACHINE-LEARNING--CHATBOT-GROUP-O/madrine/data/HIV_AIDS_Policy.pdf",
    "https://raw.githubusercontent.com/madrinejean123/MACHINE-LEARNING--CHATBOT-GROUP-O/madrine/data/UTAMU-Disability-Policy.pdf",
]

DOWNLOAD_FOLDER = "data"
os.makedirs(DOWNLOAD_FOLDER, exist_ok=True)


def download_pdfs(urls, folder):
    """Download PDF files from GitHub raw URLs."""
    downloaded = []
    for url in urls:
        filename = url.split("/")[-1]
        filepath = os.path.join(folder, filename)
        if os.path.exists(filepath):
            print(f"  ✅ Already exists: {filename}")
            downloaded.append(filepath)
            continue
        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()
            with open(filepath, "wb") as f:
                f.write(response.content)
            print(f"  ↓  Downloaded: {filename}")
            downloaded.append(filepath)
        except Exception as e:
            print(f"  ❌ Failed: {filename} — {e}")
    return downloaded


print("↓  Downloading PDFs from GitHub...")
downloaded_files = download_pdfs(GITHUB_PDF_URLS, DOWNLOAD_FOLDER)
print(f"\n   {len(downloaded_files)} files ready.")


# ── TEXT EXTRACTION ──────────────────────────────────────────────────────────────
def extract_text_from_pdf(file_path):
    text = ""
    # Attempt with PyPDF2 first for selectable text
    try:
        with open(file_path, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            for page in reader.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + " "
        # Check if PyPDF2 extracted significant text
        # A low threshold (e.g., 50 characters) can indicate it's mostly scanned or empty
        if len(text.strip()) > 50:
            print(f"  ✅ Extracted text (PyPDF2) from: {os.path.basename(file_path)}")
            return text.strip()
    except Exception as e:
        print(f"  ⚠️ PyPDF2 failed for {os.path.basename(file_path)}: {e}. Trying OCR fallback.")

    # If PyPDF2 failed or extracted very little text, try OCR
    print(f"  Attempting OCR fallback for: {os.path.basename(file_path)}")
    try:
        images = convert_from_path(file_path)
        ocr_text = ""
        for i, img in enumerate(images):
            ocr_text += pytesseract.image_to_string(img) + " "
        if ocr_text.strip():
            print(f"  ✅ Extracted text (OCR) from: {os.path.basename(file_path)}")
            return ocr_text.strip()
    except Exception as e:
        print(f"  ❌ OCR failed for {os.path.basename(file_path)}: {e}")
    return "" # Return empty string if both methods fail


# ── TEXT CLEANING ────────────────────────────────────────────────────────────────
def clean_text(text):
    ocr_fixes = {
        "har- assment": "harassment",
        "dis- ability": "disability",
        "re- port": "report",
        "com- plaint": "complaint",
    }
    for broken, fixed in ocr_fixes.items():
        text = text.replace(broken, fixed)
    text = text.replace("\n", " ")
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9.,;:()-\/ ]', '', text)
    text = re.sub(r'\(\d+\)', '', text)
    return text.strip()


# ── CHUNKING WITH OVERLAP ─────────────────────────────────────────────────────────
def chunk_text(text, max_words=150, overlap_sentences=1):
    sentences = sent_tokenize(text)
    chunks, current_chunk, current_length = [], [], 0
    for sentence in sentences:
        wc = len(sentence.split())
        if current_length + wc <= max_words:
            current_chunk.append(sentence)
            current_length += wc
        else:
            if current_chunk:
                chunks.append(" ".join(current_chunk))
            overlap = current_chunk[-overlap_sentences:] if overlap_sentences > 0 else []
            current_chunk = overlap + [sentence]
            current_length = sum(len(s.split()) for s in current_chunk)
    if current_chunk:
        chunks.append(" ".join(current_chunk))
    return chunks


# ── PROCESS ALL DOCUMENTS ─────────────────────────────────────────────────────────
def process_documents(folder_path):
    dataset = []
    pdf_files = [f for f in os.listdir(folder_path) if f.endswith(".pdf")]
    if not pdf_files:
        print("⚠️  No PDF files found in:", folder_path)
        return dataset
    for file_name in pdf_files:
        file_path = os.path.join(folder_path, file_name)
        print(f"📄 Processing: {file_name}")
        raw_text     = extract_text_from_pdf(file_path)
        cleaned_text = clean_text(raw_text)
        chunks       = chunk_text(cleaned_text)

        if not chunks:
            print(f"  ⚠️ Warning: No chunks generated for {file_name}. It might be an image-based PDF or contain no extractable text.")
            continue

        for idx, chunk in enumerate(chunks):
            dataset.append({
                "chunk_id":        f"{file_name}_chunk_{idx}",
                "source_document": file_name,
                "chunk_index":     idx,
                "text":            chunk,
                "word_count":      len(chunk.split())
            })
    return dataset


dataset = process_documents(DOWNLOAD_FOLDER)
df = pd.DataFrame(dataset)
df.to_csv("policy_chunks_dataset.csv", index=False)

print(f"\n✅ Dataset created!  Total chunks: {len(df)}  |  Documents: {df['source_document'].nunique()}")
df.head()

↓  Downloading PDFs from GitHub...
  ✅ Already exists: Makerere-Safeguarding-Policy.pdf
  ✅ Already exists: Policy-and-Regulations-Against-Sexual-Harassment-2018.pdf
  ✅ Already exists: Makerere-Policy-on-Persons-Living-With-Disabilities.pdf
  ✅ Already exists: FINAL-REVISED-NATIONAL-POLICY-ON-PWDs-2023.pdf
  ✅ Already exists: HIV_AIDS_Policy.pdf
  ✅ Already exists: UTAMU-Disability-Policy.pdf

   6 files ready.
📄 Processing: Makerere-Safeguarding-Policy.pdf
  Attempting OCR fallback for: Makerere-Safeguarding-Policy.pdf
  ✅ Extracted text (OCR) from: Makerere-Safeguarding-Policy.pdf
📄 Processing: Makerere-Policy-on-Persons-Living-With-Disabilities.pdf
  Attempting OCR fallback for: Makerere-Policy-on-Persons-Living-With-Disabilities.pdf
  ✅ Extracted text (OCR) from: Makerere-Policy-on-Persons-Living-With-Disabilities.pdf
📄 Processing: Policy-and-Regulations-Against-Sexual-Harassment-2018.pdf
  ✅ Extracted text (PyPDF2) from: Policy-and-Regulations-Against-Sexual-Harassment-2018.pdf
📄

,chunk_id,source_document,chunk_index,text,word_count
0,Makerere-Safeguarding-Policy.pdf_chunk_0,Makerere-Safeguarding-Policy.pdf,0,MAKERERE UNIVERSITY SAFEGUARDING POLICY 2024 T...,150
1,Makerere-Safeguarding-Policy.pdf_chunk_1,Makerere-Safeguarding-Policy.pdf,1,Abuse: Act or pattern of behaviour (often by s...,130
2,Makerere-Safeguarding-Policy.pdf_chunk_2,Makerere-Safeguarding-Policy.pdf,2,"Prevention: Refers to the proactive actions, s...",147
3,Makerere-Safeguarding-Policy.pdf_chunk_3,Makerere-Safeguarding-Policy.pdf,3,. Student: All individuals enrolled on differe...,140
4,Makerere-Safeguarding-Policy.pdf_chunk_4,Makerere-Safeguarding-Policy.pdf,4,. Trauma-Informed Approach: An Approach that i...,128


---
## 🔢 Step 2: Generate Embeddings

In [19]:
from sentence_transformers import SentenceTransformer

print("⏳ Loading embedding model (all-MiniLM-L6-v2)...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Embedding model loaded.")

texts = df['text'].astype(str).tolist()
print(f"\n⏳ Encoding {len(texts)} chunks...")
embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True,
    batch_size=32,
    normalize_embeddings=True
)
np.save("chunk_embeddings.npy", embeddings)
df.to_csv("chunk_metadata.csv", index=False)

print(f"\n✅ Embeddings saved! Shape: {embeddings.shape}")

⏳ Loading embedding model (all-MiniLM-L6-v2)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded.

⏳ Encoding 458 chunks...


Batches:   0%|          | 0/15 [00:00<?, ?it/s]


✅ Embeddings saved! Shape: (458, 384)


---
## 🔍 Step 3: Retrieval

In [20]:
from sklearn.metrics.pairwise import cosine_similarity

df         = pd.read_csv("chunk_metadata.csv")
embeddings = np.load("chunk_embeddings.npy")

STOP_WORDS = {'the','and','for','are','that','this','with','how',
              'what','who','can','you','was','has','have','been',
              'hello','hi','hey','please','thanks','thank'}


def keyword_filter(df, query, column='text'):
    keywords = [
        w.lower() for w in re.findall(r'\b\w+\b', query)
        if len(w) > 2 and w.lower() not in STOP_WORDS
    ]
    if not keywords:
        return df
    filtered = df[df[column].apply(lambda x: any(k in str(x).lower() for k in keywords))]
    return filtered if len(filtered) > 0 else df


def retrieve_top_k(query, model, embeddings, df, k=5, threshold=0.25):
    filtered_df         = keyword_filter(df, query)
    filtered_indices    = filtered_df.index.tolist()
    filtered_embeddings = embeddings[filtered_indices]
    query_embedding     = model.encode([query], normalize_embeddings=True)
    similarities        = cosine_similarity(query_embedding, filtered_embeddings)[0]
    sorted_idx          = np.argsort(similarities)[::-1]
    top_idx             = [i for i in sorted_idx[:k] if similarities[i] >= threshold]
    if not top_idx:
        top_idx = sorted_idx[:3]
    results = filtered_df.iloc[top_idx].copy()
    results["similarity_score"] = similarities[top_idx]
    return results


print("✅ Retrieval functions ready.")

✅ Retrieval functions ready.


---
## 💬 Step 4: Load Generation Model (flan-t5-large)

> Uses `transformers==4.40.0` where `text2text-generation` works correctly with flan-t5.

In [21]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

print("⏳ Loading generation model (flan-t5-large)...")
print("   This may take 1-2 minutes on first run.")

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-large")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large")

# Determine device: Use GPU if available, otherwise CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("✅ Generation model loaded.")

⏳ Loading generation model (flan-t5-large)...
   This may take 1-2 minutes on first run.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

✅ Generation model loaded.


In [2]:
# ── GREETING DETECTION ───────────────────────────────────────────────────────────
GREETINGS = {
    'hi', 'hello', 'hey', 'hie', 'howdy', 'good morning',
    'good afternoon', 'good evening', 'greetings', 'sup', 'wassup'
}

GREETING_RESPONSE = (
    "👋 Hello! Welcome to the Safeguarding Companion.\n\n"
    "I am here to help you understand university policies on safeguarding, "
    "disability rights, sexual harassment, and more — in simple, easy language.\n\n"
    "You can ask me things like:\n"
    "  • How do I report harassment?\n"
    "  • What rights do students with disabilities have?\n"
    "  • How do I file a complaint?\n\n"
    "What would you like to know? 😊"
)

def is_greeting(text):
    """Return True if the input is just a greeting with no real question."""
    cleaned = text.strip().lower().rstrip('!.,?')
    words   = cleaned.split()
    return cleaned in GREETINGS or (len(words) <= 3 and words[0] in GREETINGS)


# ── ANSWER GENERATION ────────────────────────────────────────────────────────────
def generate_answer(query, retrieved):
    """Generate a simplified plain-English answer from retrieved policy chunks."""
    if retrieved.empty:
        return (
            "I could not find specific information about that in the policy documents. "
            "Please try rephrasing your question or contact the Gender Mainstreaming "
            "Directorate directly for assistance."
        )

    top_chunks    = retrieved.head(3)
    context_parts = []
    for _, row in top_chunks.iterrows():
        source = (
            row['source_document']
            .replace('.pdf', '')
            .replace('-', ' ')
            .replace('_', ' ')
        )
        context_parts.append(f"[{source}]: {row['text']}")
    context = "\n\n".join(context_parts)

    prompt = (
        "You are a helpful assistant explaining university policies in simple, "
        "friendly language for students including those with disabilities.\n\n"
        "Using ONLY the policy context below, answer the question with 3 to 5 short bullet points. "
        "Use plain simple English. Avoid legal jargon. Be friendly and clear. "
        "Each bullet point should be one sentence.\n\n"
        f"Policy Context:\n{context}\n\n"
        f"Question: {query}\n\n"
        "Simple answer:"
    )

    # Tokenize the prompt and move to the model's device
    inputs = tokenizer(
        prompt,
        max_length=512, # A common max input length for T5 models
        truncation=True,
        return_tensors="pt"
    ).to(model.device)

    # Generate the answer
    outputs = model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=500, # Increased from 250 to 500
        do_sample=False
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if len(answer.split()) < 5:
        answer = (
            "The policy documents have relevant information but I could not generate "
            "a clear summary. Please review the sources listed below or contact the "
            "Gender Mainstreaming Directorate for help."
        )
    return answer


# ── FORMAT BULLET POINTS ──────────────────────────────────────────────────────────
def format_as_bullets(answer):
    """Ensure the answer displays as bullet points."""
    if '\n' in answer or answer.startswith('•') or answer.startswith('-') or answer.startswith('*'):
        return answer
    sentences = [s.strip() for s in answer.split('.') if len(s.strip()) > 10]
    return '\n'.join(f"• {s}." for s in sentences)


print("✅ Answer generation functions ready.")

✅ Answer generation functions ready.


---
## 🧪 Step 5: Test the Full RAG Pipeline

In [3]:
def rag_answer(query, show_sources=True):
    """Full RAG pipeline — retrieve + generate + display."""
    print(f"\n{'='*60}")
    print(f"❓ Question: {query}")
    print(f"{'='*60}")

    if is_greeting(query):
        print(GREETING_RESPONSE)
        return GREETING_RESPONSE

    retrieved = retrieve_top_k(query, embedding_model, embeddings, df, k=5)
    answer    = generate_answer(query, retrieved)
    formatted = format_as_bullets(answer)

    print(f"\n✅ Answer:\n{formatted}")

    if show_sources and not retrieved.empty:
        print(f"\n📎 Sources:")
        for src in retrieved['source_document'].unique():
            clean = src.replace('.pdf','').replace('-',' ').replace('_',' ')
            print(f"   • {clean}")

    print(f"{'='*60}\n")
    return formatted


# Test questions
rag_answer("hello")
rag_answer("how do I file a complaint?")
rag_answer("what rights do students with disabilities have?")
rag_answer("how do I report sexual harassment?")


❓ Question: hello
👋 Hello! Welcome to the Safeguarding Companion.

I am here to help you understand university policies on safeguarding, disability rights, sexual harassment, and more — in simple, easy language.

You can ask me things like:
  • How do I report harassment?
  • What rights do students with disabilities have?
  • How do I file a complaint?

What would you like to know? 😊

❓ Question: how do I file a complaint?


NameError: name 'retrieve_top_k' is not defined

---
## 🖥️ Step 6: Gradio UI — Voice Input + Text-to-Speech Output

In [24]:
import gradio as gr
import tempfile
from gtts import gTTS


# ── TEXT-TO-SPEECH ────────────────────────────────────────────────────────────────
def text_to_speech(text):
    """Convert answer text to an MP3 audio file."""
    try:
        clean = text.replace('•', '').replace('*', '').replace('-', '')
        tts   = gTTS(text=clean, lang='en', slow=False)
        tmp   = tempfile.NamedTemporaryFile(delete=False, suffix='.mp3')
        tts.save(tmp.name)
        return tmp.name
    except Exception as e:
        print(f"TTS error: {e}")
        return None


# ── VOICE TRANSCRIPTION ───────────────────────────────────────────────────────────
def transcribe_audio(audio_path):
    """Transcribe voice input using Whisper-tiny."""
    try:
        from transformers import pipeline as hf_pipeline
        transcriber = hf_pipeline(
            "automatic-speech-recognition",
            model="openai/whisper-tiny"
        )
        result = transcriber(audio_path)
        return result["text"].strip()
    except Exception as e:
        print(f"Transcription error: {e}")
        return ""


# ── MAIN HANDLER ──────────────────────────────────────────────────────────────────
def handle_query(text_query, audio_query):
    """Handle text or voice input, return answer + audio + sources."""
    query = ""
    if audio_query is not None:
        query = transcribe_audio(audio_query)
        print(f"🎙️ Transcribed: {query}")
    if not query:
        query = text_query or ""

    if not query.strip():
        msg = "Please type or speak your question."
        return msg, text_to_speech(msg), ""

    if is_greeting(query):
        return GREETING_RESPONSE, text_to_speech(GREETING_RESPONSE), ""

    retrieved = retrieve_top_k(query, embedding_model, embeddings, df, k=5)
    answer    = generate_answer(query, retrieved)
    formatted = format_as_bullets(answer)

    sources = ""
    if not retrieved.empty:
        src_list = [
            src.replace('.pdf','').replace('-',' ').replace('_',' ')
            for src in retrieved['source_document'].unique()
        ]
        sources = "📎 Sources:\n" + "\n".join(f"  • {s}" for s in src_list)

    audio_path = text_to_speech(answer)
    return formatted, audio_path, sources


# ── GRADIO INTERFACE ──────────────────────────────────────────────────────────────
with gr.Blocks(theme=gr.themes.Soft(), title="Safeguarding Companion") as demo:

    gr.Markdown("""
    # 🛡️ Safeguarding Companion
    **Ask questions about safeguarding policies, disability rights, harassment, and reporting procedures.**
    Answers are in plain, simple English — grounded in official university policy documents.

    > 🎙️ You can **type** your question OR **record your voice**.
    > 🔊 Tap the audio player below to **listen** to the answer read aloud.
    """)

    with gr.Row():
        with gr.Column(scale=2):
            text_input = gr.Textbox(
                lines=2,
                placeholder="Type your question here...",
                label="✏️ Type your question"
            )
        with gr.Column(scale=1):
            audio_input = gr.Audio(
                sources=["microphone"],
                type="filepath",
                label="🎙️ Or speak your question"
            )

    submit_btn = gr.Button("🔍 Get Answer", variant="primary", size="lg")

    with gr.Row():
        with gr.Column(scale=3):
            text_output = gr.Textbox(
                lines=12,
                label="💬 Answer",
                interactive=False
            )
        with gr.Column(scale=1):
            sources_output = gr.Textbox(
                lines=6,
                label="📎 Policy Sources",
                interactive=False
            )

    audio_output = gr.Audio(
        label="🔊 Listen to the answer — tap play",
        type="filepath",
        interactive=False,
        autoplay=False
    )

    gr.Examples(
        examples=[
            ["Hello!", None],
            ["How do I file a complaint?", None],
            ["What rights do students with disabilities have?", None],
            ["How do I report sexual harassment?", None],
            ["What support is available for persons with disabilities?", None],
            ["What is the university HIV/AIDS policy?", None],
        ],
        inputs=[text_input, audio_input],
        label="💡 Example questions — click any to try"
    )

    submit_btn.click(
        fn=handle_query,
        inputs=[text_input, audio_input],
        outputs=[text_output, audio_output, sources_output]
    )

    text_input.submit(
        fn=handle_query,
        inputs=[text_input, audio_input],
        outputs=[text_output, audio_output, sources_output]
    )


# share=True gives a public link on Colab
demo.launch(share=True)

/tmp/ipykernel_7171/3865259889.py:70: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Safeguarding Companion") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2d035d68985a25b65b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## 📋 Document Summary

In [ ]:
print("📂 Processed Policy Documents:")
print("-" * 50)
for doc in df['source_document'].unique():
    count = len(df[df['source_document'] == doc])
    clean = doc.replace('.pdf','').replace('-',' ').replace('_',' ')
    print(f"  ✅ {clean}  ({count} chunks)")
print("-" * 50)
print(f"  Total: {len(df)} chunks across {df['source_document'].nunique()} documents")